## 02 — Clean Baseline Pipeline

Same baseline as `01_initial_data_exploration.ipynb`, but all logic lives in `src/solarops/pipeline.py`.

**Goal:** reproduce v0 / v1 results in ~12 lines. No exploration here — exploration stays in `01`.

### Setup
Auto-reload so edits to `pipeline.py` take effect without restarting the kernel.

In [1]:
%load_ext autoreload
%autoreload 2

from pathlib import Path
import pandas as pd
from sklearn.dummy import DummyRegressor
from sklearn.linear_model import LinearRegression

import solarops.pipeline
from solarops.pipeline import (
    load_raw, align_15m, split_time, impute_features, evaluate,
)

### Data location
`DATA_DIR` is inherited from cell 3 of notebook 01 — but in a fresh kernel it doesn't exist. Define once here.

In [2]:
PROJECT_ROOT = Path.cwd()
while not (PROJECT_ROOT / "pyproject.toml").exists():
    if PROJECT_ROOT.parent == PROJECT_ROOT:
        raise FileNotFoundError("SolarOps project root not found.")
    PROJECT_ROOT = PROJECT_ROOT.parent

DATA_DIR = PROJECT_ROOT / "data/raw/pvdaq/2023-solar-data-prize/system-2107"
print(DATA_DIR)

/Users/apple/Desktop/SolarOps/data/raw/pvdaq/2023-solar-data-prize/system-2107


### 1. Load raw

In [3]:
raw = load_raw(DATA_DIR)
for k, v in raw.items():
    print(f"{k:12} | rows={len(v):>7,} | dtype={v['measured_on'].dtype}")

meter        | rows=240,062 | dtype=datetime64[us]
environment  | rows=206,008 | dtype=datetime64[us]
irradiance   | rows=531,019 | dtype=datetime64[us]


### 2. Align to 15-min grid

In [4]:
df = align_15m(raw)
print("shape:", df.shape)
print("window:", df["measured_on"].min(), "→", df["measured_on"].max())
print("\nMissing %:")
print((df.isna().mean() * 100).round(2))

shape: (207140, 5)
window: 2017-12-01 00:00:00 → 2023-10-31 23:45:00

Missing %:
measured_on                                   0.00
meter_revenue_grade_ac_output_meter_149578    0.00
ambient_temperature_o_149575                  0.75
wind_speed_o_149576                           0.70
poa_irradiance_o_149574                       5.50
dtype: float64


### 3. Chronological split
Train < 2022-07-01 ≤ Test. No random shuffling — this is a time series.

In [5]:
train, test = split_time(df, "2022-07-01")
print(f"train: {len(train):,} | test: {len(test):,}")

train: 160,396 | test: 46,744


### 4. Impute features
Median imputer **fit on train only**, applied to train and test. No leakage.

In [6]:
X_train, X_test, y_train, y_test = impute_features(train, test)
print("X_train:", X_train.shape, "| X_test:", X_test.shape)
print("Any NaN:", X_train.isna().any().any(), X_test.isna().any().any())

X_train: (160396, 3) | X_test: (46744, 3)
Any NaN: False False


### 5. Models + evaluation
v0 = dummy ruler. v1 = linear regression. Both scored on the same test set.

In [7]:
v0 = DummyRegressor(strategy="mean").fit(X_train, y_train)
v1 = LinearRegression().fit(X_train, y_train)

results = pd.DataFrame([
    evaluate(v0, X_test, y_test, "v0 dummy"),
    evaluate(v1, X_test, y_test, "v1 linear regression"),
]).round(3)

results["MAE_improvement_vs_v0_%"] = (
    (results.loc[0, "MAE"] - results["MAE"]) / results.loc[0, "MAE"] * 100
).round(1)

results

,model,MAE,RMSE,R2,MAE_improvement_vs_v0_%
0,v0 dummy,196.202,231.624,-0.002,0.0
1,v1 linear regression,38.755,66.631,0.917,80.2


## 6. Reflection

- v0 MAE ≈ 196 kW (the floor).
- v1 MAE ≈ 38.8 kW → **80% better**, R² ≈ 0.917. Data has signal.
- Coefficients match PV physics: POA positive, temperature negative (derating).
- Test window Jul 2022 – Oct 2023 doesn't cover spring → next step is per-month error, not just overall.
- **Next:** diagnostics (`03_diagnostics.ipynb`) — residuals vs hour, vs POA, vs month.